# Hidden Markov Model Regime Detection for Financial Time Series

This notebook implements the second regime detection method for the thesis:

**Regime-Conditioned Multivariate Motif Discovery In Financial Time Series: A Reproducible Empirical Benchmark Under Nonstationarity**

Financial time series are nonstationary. Market behaviour changes across calm, normal, and stressed periods, and those changes can alter later motif-discovery experiments.

The previous notebook used volatility-quantile regimes. Quantile regimes are transparent, but threshold-based. This notebook uses a **Hidden Markov Model (HMM)** as the data-driven probabilistic regime method. The HMM infers hidden low, medium, and high-volatility market states from observed return and volatility-related features.

The thesis compares motif mining in two settings:

1. regime-agnostic discovery on the full series
2. regime-conditioned discovery inside detected regimes

This notebook intentionally does **not** implement motif discovery, Matrix Profile, or LoCoMotif. It only prepares clean HMM regime labels for BTCUSDT and ETHUSDT.


## 1. Imports and Configuration

The configuration mirrors the first regime notebook where possible: target crypto assets, a 2025 analysis window, and rolling windows inferred from data frequency. HMM-specific settings are kept explicit for reproducibility.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
try:
    from sklearn.metrics import adjusted_rand_score
except Exception:
    adjusted_rand_score = None

try:
    from hmmlearn.hmm import GaussianHMM
except ImportError as exc:
    raise ImportError("hmmlearn is required for this notebook. Install it with: pip install hmmlearn") from exc

PROJECT_ROOT = Path(r"C:\Users\learn\OneDrive\Desktop\Final Masters Thesis")
DATA_DIR = PROJECT_ROOT / "final_dataset"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "Regime Studies"
QUANTILE_RESULT_DIR = PROJECT_ROOT / "reports" / "results" / "regime_studies" / "01_volatility_quantile_regimes"
RESULT_DIR = PROJECT_ROOT / "reports" / "results" / "regime_studies" / "02_hmm_regimes"

FIGURE_DIR = RESULT_DIR / "figures"
TABLE_DIR = RESULT_DIR / "tables"
REGIME_LABEL_DIR = RESULT_DIR / "regime_labels"
CONFIG_DIR = RESULT_DIR / "config"
LOG_DIR = RESULT_DIR / "logs"
MODEL_DIR = RESULT_DIR / "models"

for directory in [FIGURE_DIR, TABLE_DIR, REGIME_LABEL_DIR, CONFIG_DIR, LOG_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TARGET_SYMBOLS = ["BTCUSDT", "ETHUSDT"]
USE_YEAR_FILTER = True
YEAR_TO_ANALYZE = 2025

N_HMM_STATES = 3
HMM_COVARIANCE_TYPE = "full"
HMM_N_ITER = 500
HMM_RANDOM_STATE = 42
HMM_TOL = 1e-4

ROLLING_WINDOW_OVERRIDE = None

HMM_FEATURES_BASE = ["log_return", "abs_log_return", "rolling_realized_vol"]
HMM_FEATURES = [
    "log_return",
    "abs_log_return",
    "rolling_realized_vol",
    "rolling_mean_return",
    "rolling_abs_return_mean",
]

SUPPORTED_EXTENSIONS = {".parquet", ".csv", ".feather", ".pkl", ".pickle"}
REGIME_ORDER = ["low_vol", "medium_vol", "high_vol"]
REGIME_ID_TO_LABEL = {0: "low_vol", 1: "medium_vol", 2: "high_vol"}
REGIME_COLORS = {
    "low_vol": "tab:green",
    "medium_vol": "tab:orange",
    "high_vol": "tab:red",
    "insufficient_history": "lightgray",
}

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print(f"Data directory: {DATA_DIR}")
print(f"Result directory: {RESULT_DIR}")


Data directory: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\final_dataset
Result directory: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\02_hmm_regimes


## 2. Data Discovery and Loading

The loader searches recursively for supported tabular files and ranks likely crypto OHLCV sources first. It supports both separate symbol files and combined files with a symbol column. The final output is an `asset_data` dictionary keyed by asset symbol.


In [2]:
def discover_candidate_files(data_dir):
    """Find supported data files recursively and rank likely crypto OHLCV sources first."""
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory does not exist: {data_dir}")

    records = []
    preferred_keywords = ["btcusdt", "ethusdt", "btc", "eth", "crypto", "final", "clean", "processed"]
    avoid_keywords = ["metadata", "readme", "requirements", "missing_data", "download_log", "inventory"]

    for path in data_dir.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        name_lower = path.name.lower()
        full_lower = str(path).lower()
        score = 0
        reasons = []
        if path.suffix.lower() == ".parquet":
            score += 10
            reasons.append("parquet")
        if "processed" in full_lower:
            score += 25
            reasons.append("processed")
        if "crypto" in full_lower:
            score += 18
            reasons.append("crypto")
        if "features" in full_lower:
            score += 8
            reasons.append("features_available")
        if "raw" in full_lower:
            score -= 25
            reasons.append("raw_lower_priority")
        if "1h" in name_lower or "\\1h\\" in full_lower or "/1h/" in full_lower:
            score += 22
            reasons.append("1h_preferred")
        elif "15m" in name_lower or "\\15m\\" in full_lower or "/15m/" in full_lower:
            score += 14
            reasons.append("15m")
        elif "5m" in name_lower or "\\5m\\" in full_lower or "/5m/" in full_lower:
            score += 10
            reasons.append("5m")
        elif "1d" in name_lower or "\\1d\\" in full_lower or "/1d/" in full_lower:
            score += 7
            reasons.append("1d")
        elif "1m" in name_lower or "\\1m\\" in full_lower or "/1m/" in full_lower:
            score += 2
            reasons.append("1m_large_but_valid")

        for keyword in preferred_keywords:
            if keyword in full_lower:
                score += 4
                reasons.append(f"keyword:{keyword}")
        for keyword in avoid_keywords:
            if keyword in full_lower:
                score -= 100
                reasons.append(f"avoid:{keyword}")

        records.append({
            "path": path,
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "discovery_score": score,
            "discovery_reasons": ";".join(reasons),
        })

    candidates = pd.DataFrame(records)
    if candidates.empty:
        expected = ", ".join(sorted(SUPPORTED_EXTENSIONS))
        raise FileNotFoundError(f"No candidate files found under {data_dir}. Expected extensions: {expected}")
    return candidates.sort_values(["discovery_score", "size_bytes"], ascending=[False, True]).reset_index(drop=True)


def preview_file(path, max_rows=5000):
    path = Path(path)
    try:
        if path.suffix.lower() == ".csv":
            return pd.read_csv(path, nrows=max_rows)
        if path.suffix.lower() == ".feather":
            return pd.read_feather(path).head(max_rows)
        if path.suffix.lower() in {".pkl", ".pickle"}:
            obj = pd.read_pickle(path)
            return obj.head(max_rows) if isinstance(obj, pd.DataFrame) else pd.DataFrame()
        if path.suffix.lower() == ".parquet":
            try:
                import pyarrow.parquet as pq
                parquet_file = pq.ParquetFile(path)
                if parquet_file.num_row_groups > 0:
                    return parquet_file.read_row_group(0).to_pandas().head(max_rows)
            except Exception:
                pass
            return pd.read_parquet(path).head(max_rows)
    except Exception as exc:
        warnings.warn(f"Could not preview {path}: {exc}")
    return pd.DataFrame()


def read_table_file(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".feather":
        return pd.read_feather(path)
    if path.suffix.lower() in {".pkl", ".pickle"}:
        obj = pd.read_pickle(path)
        if not isinstance(obj, pd.DataFrame):
            raise TypeError(f"Pickle did not contain a DataFrame: {path}")
        return obj
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported extension: {path.suffix}")


def detect_timestamp_column(df):
    lower_map = {str(col).lower(): col for col in df.columns}
    for name in ["timestamp", "datetime", "date", "open_time", "close_time", "time"]:
        if name in lower_map:
            return lower_map[name]
    for col in df.columns:
        if "time" in str(col).lower() or "date" in str(col).lower():
            return col
    return None


def detect_close_column(df):
    lower_map = {str(col).lower(): col for col in df.columns}
    for name in ["close", "adj_close", "adjusted_close", "last", "price"]:
        if name in lower_map:
            return lower_map[name]
    for col in df.columns:
        if "close" in str(col).lower():
            return col
    return None


def detect_volume_column(df):
    lower_map = {str(col).lower(): col for col in df.columns}
    for name in ["volume", "base_volume", "quote_volume"]:
        if name in lower_map:
            return lower_map[name]
    for col in df.columns:
        if "volume" in str(col).lower():
            return col
    return None


def detect_symbol_column(df):
    lower_map = {str(col).lower(): col for col in df.columns}
    for name in ["symbol", "asset", "ticker"]:
        if name in lower_map:
            return lower_map[name]
    return None


def infer_symbol_from_path(path, target_symbols):
    text = str(path).lower()
    for symbol in target_symbols:
        if symbol.lower() in text:
            return symbol
    return None


def standardize_asset_frame(df, asset, timestamp_col, close_col, volume_col=None, symbol_col=None):
    keep = {timestamp_col: "timestamp", close_col: "close"}
    if volume_col is not None:
        keep[volume_col] = "volume"
    if symbol_col is not None:
        keep[symbol_col] = "symbol"
    out = df[list(keep.keys())].rename(columns=keep).copy()
    if "symbol" not in out.columns:
        out["symbol"] = asset
    return out


def load_asset_data(data_dir, target_symbols):
    candidates = discover_candidate_files(data_dir)
    asset_frames = {}
    input_files_used = {}
    log_rows = []

    for _, row in candidates.iterrows():
        path = Path(row["path"])
        preview = preview_file(path)
        timestamp_col = detect_timestamp_column(preview) if not preview.empty else None
        close_col = detect_close_column(preview) if not preview.empty else None
        volume_col = detect_volume_column(preview) if not preview.empty else None
        symbol_col = detect_symbol_column(preview) if not preview.empty else None
        inferred_symbol = infer_symbol_from_path(path, target_symbols)
        status = "previewed"
        loaded_symbols = []
        warning_message = ""

        if timestamp_col is None or close_col is None:
            status = "skipped_no_timestamp_or_close"
        else:
            if symbol_col is not None:
                observed = preview[symbol_col].astype(str).str.upper().unique().tolist()
                candidate_targets = [s for s in target_symbols if s in observed]
            elif inferred_symbol is not None:
                candidate_targets = [inferred_symbol]
            else:
                candidate_targets = []
            candidate_targets = [s for s in candidate_targets if s not in asset_frames]

            if candidate_targets:
                try:
                    full_df = read_table_file(path)
                    ts = detect_timestamp_column(full_df)
                    close = detect_close_column(full_df)
                    vol = detect_volume_column(full_df)
                    sym = detect_symbol_column(full_df)
                    for symbol in candidate_targets:
                        source = full_df
                        if sym is not None:
                            source = full_df[full_df[sym].astype(str).str.upper() == symbol].copy()
                        if source.empty:
                            continue
                        asset_frames[symbol] = standardize_asset_frame(source, symbol, ts, close, vol, sym)
                        input_files_used[symbol] = str(path)
                        loaded_symbols.append(symbol)
                    status = "loaded" if loaded_symbols else "previewed_no_matching_rows"
                except Exception as exc:
                    status = "failed_to_load"
                    warning_message = str(exc)
                    warnings.warn(f"Failed to load {path}: {exc}")
            else:
                status = "previewed_not_selected"

        log_rows.append({
            "path": str(path),
            "file_name": row["file_name"],
            "extension": row["extension"],
            "size_bytes": row["size_bytes"],
            "discovery_score": row["discovery_score"],
            "discovery_reasons": row["discovery_reasons"],
            "status": status,
            "timestamp_column": timestamp_col,
            "close_column": close_col,
            "volume_column": volume_col,
            "symbol_column": symbol_col,
            "inferred_symbol_from_path": inferred_symbol,
            "loaded_symbols": ",".join(loaded_symbols),
            "warning": warning_message,
        })
        if all(symbol in asset_frames for symbol in target_symbols):
            break

    missing_assets = [symbol for symbol in target_symbols if symbol not in asset_frames]
    if missing_assets:
        warnings.warn(f"Missing target assets after discovery: {missing_assets}. Continuing with available assets.")
    if not asset_frames:
        expected = ", ".join(sorted(SUPPORTED_EXTENSIONS))
        raise FileNotFoundError(f"No usable target asset data found under {data_dir}. Expected extensions: {expected}")
    return asset_frames, input_files_used, pd.DataFrame(log_rows)


asset_data, input_files_used, discovery_log = load_asset_data(DATA_DIR, TARGET_SYMBOLS)
discovery_log.to_csv(LOG_DIR / "data_discovery_log.csv", index=False)

print("Assets loaded:", list(asset_data.keys()))
for asset, df in asset_data.items():
    print(f"{asset}: {len(df):,} rows from {input_files_used[asset]}")
display(discovery_log.head(20))


Assets loaded: ['ETHUSDT', 'BTCUSDT']
ETHUSDT: 52,577 rows from C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\final_dataset\processed\crypto\1h\ETHUSDT_1h_2020_2025.parquet
BTCUSDT: 52,577 rows from C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\final_dataset\processed\crypto\1h\BTCUSDT_1h_2020_2025.parquet


,path,file_name,extension,size_bytes,discovery_score,discovery_reasons,status,timestamp_column,close_column,volume_column,symbol_column,inferred_symbol_from_path,loaded_symbols,warning
0,C:\Users\learn\OneDrive\Desktop\Final Masters ...,ETHUSDT_1h_2020_2025.parquet,.parquet,4515758,95,parquet;processed;crypto;1h_preferred;keyword:...,loaded,timestamp,close,volume,None,ETHUSDT,ETHUSDT,
1,C:\Users\learn\OneDrive\Desktop\Final Masters ...,BTCUSDT_1h_2020_2025.parquet,.parquet,4779460,95,parquet;processed;crypto;1h_preferred;keyword:...,loaded,timestamp,close,volume,None,BTCUSDT,BTCUSDT,


## 3. Data Cleaning

Each asset is timestamp-normalized, sorted, de-duplicated, filtered to positive close prices, and restricted to 2025 when that year is available. If 2025 is unavailable, the full available range is used with a warning.


In [3]:
def convert_timestamp(series):
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        median_value = numeric.dropna().median()
        if pd.isna(median_value):
            return pd.to_datetime(series, errors="coerce", utc=True)
        if median_value > 1e17:
            unit = "ns"
        elif median_value > 1e14:
            unit = "us"
        elif median_value > 1e11:
            unit = "ms"
        else:
            unit = "s"
        return pd.to_datetime(numeric, errors="coerce", unit=unit, utc=True)
    return pd.to_datetime(series, errors="coerce", utc=True)


def infer_sampling_frequency(timestamps):
    clean_times = pd.Series(timestamps).dropna().sort_values()
    diffs = clean_times.diff().dropna()
    if diffs.empty:
        return {"median_delta": pd.NaT, "frequency_label": "unknown", "periods_per_year": np.nan}
    median_delta = diffs.median()
    minutes = median_delta / pd.Timedelta(minutes=1)
    if minutes <= 2:
        return {"median_delta": median_delta, "frequency_label": "minute_level", "periods_per_year": 365 * 24 * 60}
    if minutes <= 10:
        return {"median_delta": median_delta, "frequency_label": "five_minute_level", "periods_per_year": 365 * 24 * 12}
    if minutes <= 20:
        return {"median_delta": median_delta, "frequency_label": "fifteen_minute_level", "periods_per_year": 365 * 24 * 4}
    if minutes <= 90:
        return {"median_delta": median_delta, "frequency_label": "hourly", "periods_per_year": 365 * 24}
    if minutes <= 60 * 30:
        return {"median_delta": median_delta, "frequency_label": "daily_or_intraday_gap", "periods_per_year": 365}
    return {"median_delta": median_delta, "frequency_label": "low_frequency_or_irregular", "periods_per_year": np.nan}


def clean_asset_frame(df, asset, use_year_filter=True, year_to_analyze=2025):
    cleaned = df.copy()
    cleaned["timestamp"] = convert_timestamp(cleaned["timestamp"])
    cleaned["close"] = pd.to_numeric(cleaned["close"], errors="coerce")
    if "volume" in cleaned.columns:
        cleaned["volume"] = pd.to_numeric(cleaned["volume"], errors="coerce")
    if "symbol" not in cleaned.columns:
        cleaned["symbol"] = asset

    missing_close_before = int(cleaned["close"].isna().sum())
    rows_raw = len(cleaned)
    cleaned = cleaned.dropna(subset=["timestamp", "close"])
    cleaned = cleaned[cleaned["close"] > 0].copy()
    cleaned = cleaned.sort_values("timestamp").drop_duplicates("timestamp", keep="last").reset_index(drop=True)
    full_start = cleaned["timestamp"].min()
    full_end = cleaned["timestamp"].max()
    used_year_filter = False
    if use_year_filter:
        year_mask = cleaned["timestamp"].dt.year == year_to_analyze
        if year_mask.any():
            cleaned = cleaned.loc[year_mask].reset_index(drop=True)
            used_year_filter = True
        else:
            warnings.warn(f"{asset}: year {year_to_analyze} was not found; using full range {full_start} to {full_end}.")

    frequency_info = infer_sampling_frequency(cleaned["timestamp"])
    report = {
        "asset": asset,
        "n_rows_raw": rows_raw,
        "n_rows_after_cleaning": len(cleaned),
        "start_time": cleaned["timestamp"].min(),
        "end_time": cleaned["timestamp"].max(),
        "missing_close_before_cleaning": missing_close_before,
        "missing_close_after_cleaning": int(cleaned["close"].isna().sum()),
        "median_delta": frequency_info["median_delta"],
        "frequency_label": frequency_info["frequency_label"],
        "periods_per_year": frequency_info["periods_per_year"],
        "used_year_filter": used_year_filter,
    }
    return cleaned, report


cleaned_asset_data = {}
cleaning_reports = []
frequency_info_by_asset = {}
for asset, df in asset_data.items():
    cleaned, report = clean_asset_frame(df, asset, USE_YEAR_FILTER, YEAR_TO_ANALYZE)
    cleaned_asset_data[asset] = cleaned
    cleaning_reports.append(report)
    frequency_info_by_asset[asset] = {
        "frequency_label": report["frequency_label"],
        "periods_per_year": report["periods_per_year"],
        "median_delta": report["median_delta"],
    }

cleaning_report_df = pd.DataFrame(cleaning_reports)
cleaning_report_df.to_csv(TABLE_DIR / "data_cleaning_report_by_asset.csv", index=False)
display(cleaning_report_df)


,asset,n_rows_raw,n_rows_after_cleaning,start_time,end_time,missing_close_before_cleaning,missing_close_after_cleaning,median_delta,frequency_label,periods_per_year,used_year_filter
0,ETHUSDT,52577,8760,2025-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,0,0 days 01:00:00,hourly,8760,True
1,BTCUSDT,52577,8760,2025-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,0,0 days 01:00:00,hourly,8760,True


## 4. Feature Engineering

The HMM observes return and volatility-related features. Every rolling feature uses a trailing window over current and historical observations only. No centered windows or future values are used.


In [4]:
def default_rolling_window(frequency_label):
    if frequency_label == "minute_level":
        return 60
    if frequency_label == "hourly":
        return 24
    if frequency_label == "daily_or_intraday_gap":
        return 20
    return 60


def add_hmm_features(df, frequency_info, rolling_window_override=None):
    featured = df.copy()
    featured["log_close"] = np.log(featured["close"])
    featured["log_return"] = featured["log_close"].diff()
    featured["abs_log_return"] = featured["log_return"].abs()
    window = rolling_window_override or default_rolling_window(frequency_info["frequency_label"])
    featured["rolling_realized_vol"] = featured["log_return"].rolling(window=window, min_periods=window).std()
    featured["rolling_mean_return"] = featured["log_return"].rolling(window=window, min_periods=window).mean()
    featured["rolling_abs_return_mean"] = featured["abs_log_return"].rolling(window=window, min_periods=window).mean()
    if "volume" in featured.columns:
        featured["volume"] = pd.to_numeric(featured["volume"], errors="coerce")
        featured["log_volume"] = np.log1p(featured["volume"].clip(lower=0))
        featured["volume_log_change"] = featured["log_volume"].diff()
        featured["rolling_volume_mean"] = featured["log_volume"].rolling(window=window, min_periods=window).mean()
        featured["rolling_volume_std"] = featured["log_volume"].rolling(window=window, min_periods=window).std()
        featured["rolling_volume_z"] = (featured["log_volume"] - featured["rolling_volume_mean"]) / featured["rolling_volume_std"]
    return featured, int(window)


def select_valid_hmm_features(df, requested_features):
    valid_features = []
    dropped = []
    for feature in requested_features:
        if feature not in df.columns:
            dropped.append((feature, "missing"))
            continue
        numeric = pd.to_numeric(df[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if numeric.notna().sum() == 0:
            dropped.append((feature, "all_missing_or_infinite"))
            continue
        if numeric.nunique(dropna=True) <= 1:
            dropped.append((feature, "constant"))
            continue
        valid_features.append(feature)
    if dropped:
        warnings.warn(f"Dropped invalid HMM features: {dropped}")
    return valid_features, dropped


featured_asset_data = {}
rolling_windows_used = {}
features_used_by_asset = {}
feature_drop_log = []
for asset, df in cleaned_asset_data.items():
    featured, window = add_hmm_features(df, frequency_info_by_asset[asset], ROLLING_WINDOW_OVERRIDE)
    optional_features = []
    if "volume_log_change" in featured.columns:
        optional_features.extend(["volume_log_change", "rolling_volume_z"])
    valid_features, dropped = select_valid_hmm_features(featured, HMM_FEATURES + optional_features)
    featured_asset_data[asset] = featured
    rolling_windows_used[asset] = window
    features_used_by_asset[asset] = valid_features
    for feature, reason in dropped:
        feature_drop_log.append({"asset": asset, "feature": feature, "reason": reason})
    usable_rows = featured[valid_features].replace([np.inf, -np.inf], np.nan).dropna().shape[0]
    print(f"{asset}: rolling window = {window}; HMM features = {valid_features}; usable rows = {usable_rows:,}")

pd.DataFrame(feature_drop_log).to_csv(LOG_DIR / "hmm_feature_drop_log.csv", index=False)


ETHUSDT: rolling window = 24; HMM features = ['log_return', 'abs_log_return', 'rolling_realized_vol', 'rolling_mean_return', 'rolling_abs_return_mean', 'volume_log_change', 'rolling_volume_z']; usable rows = 8,736
BTCUSDT: rolling window = 24; HMM features = ['log_return', 'abs_log_return', 'rolling_realized_vol', 'rolling_mean_return', 'rolling_abs_return_mean', 'volume_log_change', 'rolling_volume_z']; usable rows = 8,736


## 5. How Hidden Markov Models Represent Regimes

A Hidden Markov Model assumes that the market has hidden states. We do not directly observe the state. Instead, we observe returns and volatility features, and the model infers which hidden state most likely generated each observation.

There are two important parts:

1. **Emissions**: each state has a distribution over observed features. One state may generate low-volatility returns, while another may generate high-volatility returns.
2. **Transitions**: the transition matrix describes how likely the market is to remain in the same state or switch into another state. High diagonal probabilities indicate persistent regimes.

This is useful because HMMs can capture volatility clustering, regime persistence, and data-driven state changes instead of relying on fixed thresholds.

Limitations matter. Raw HMM state labels are arbitrary: state `0` does not automatically mean low volatility. After fitting, states must be sorted by average rolling realized volatility. HMM results can also be sensitive to feature scaling, the number of states, and the Gaussian emission assumption.


## 6. HMM Regime Fitting

The function below fits a Gaussian HMM, predicts raw hidden states, computes posterior state probabilities, and interprets raw states by sorting them from lowest to highest average rolling realized volatility.


In [5]:
def fit_hmm_regimes(df, feature_cols, n_states=3, covariance_type="full", n_iter=500, random_state=42, tol=1e-4, scaler_type="standard"):
    if not feature_cols:
        raise ValueError("No valid HMM feature columns were provided.")
    fitted = df.copy()
    for col in feature_cols:
        fitted[col] = pd.to_numeric(fitted[col], errors="coerce")
    feature_frame = fitted[feature_cols].replace([np.inf, -np.inf], np.nan)
    valid_mask = feature_frame.notna().all(axis=1)
    valid_index = fitted.index[valid_mask]
    valid_features = feature_frame.loc[valid_mask]
    if len(valid_features) < max(100, n_states * 20):
        raise ValueError(f"Not enough valid observations for HMM fitting: {len(valid_features)}")

    scaler = RobustScaler() if scaler_type == "robust" else StandardScaler()
    X = scaler.fit_transform(valid_features)
    model = GaussianHMM(
        n_components=n_states,
        covariance_type=covariance_type,
        n_iter=n_iter,
        random_state=random_state,
        tol=tol,
        min_covar=1e-6,
    )
    model.fit(X)
    raw_states = model.predict(X)
    probabilities = model.predict_proba(X)

    fitted["raw_hmm_state"] = pd.Series(pd.NA, index=fitted.index, dtype="Int64")
    fitted["hmm_regime_id"] = pd.Series(pd.NA, index=fitted.index, dtype="Int64")
    fitted["hmm_regime_label"] = "insufficient_history"
    fitted["hmm_state_probability_max"] = np.nan
    for state in range(n_states):
        fitted[f"hmm_prob_state_{state}"] = np.nan

    fitted.loc[valid_index, "raw_hmm_state"] = raw_states
    fitted.loc[valid_index, "hmm_state_probability_max"] = probabilities.max(axis=1)
    for state in range(n_states):
        fitted.loc[valid_index, f"hmm_prob_state_{state}"] = probabilities[:, state]

    state_vol = (
        pd.DataFrame({"raw_hmm_state": raw_states, "rolling_realized_vol": fitted.loc[valid_index, "rolling_realized_vol"].to_numpy()})
        .groupby("raw_hmm_state")["rolling_realized_vol"]
        .mean()
        .sort_values()
    )
    raw_to_interpreted = {int(raw_state): int(rank) for rank, raw_state in enumerate(state_vol.index)}
    raw_to_label = {raw_state: REGIME_ID_TO_LABEL.get(regime_id, f"state_{regime_id}") for raw_state, regime_id in raw_to_interpreted.items()}
    fitted.loc[valid_index, "hmm_regime_id"] = [raw_to_interpreted[int(state)] for state in raw_states]
    fitted.loc[valid_index, "hmm_regime_label"] = [raw_to_label[int(state)] for state in raw_states]

    valid_labeled = fitted.loc[valid_index].copy()
    rows = []
    for raw_state, group in valid_labeled.groupby("raw_hmm_state"):
        raw_state_int = int(raw_state)
        rows.append({
            "raw_hmm_state": raw_state_int,
            "interpreted_regime_id": raw_to_interpreted[raw_state_int],
            "interpreted_regime_label": raw_to_label[raw_state_int],
            "n_observations": int(len(group)),
            "percentage_observations": float(len(group) / len(valid_labeled) * 100),
            "mean_log_return": float(group["log_return"].mean()),
            "std_log_return": float(group["log_return"].std()),
            "mean_abs_log_return": float(group["abs_log_return"].mean()),
            "mean_rolling_vol": float(group["rolling_realized_vol"].mean()),
            "median_rolling_vol": float(group["rolling_realized_vol"].median()),
            "min_rolling_vol": float(group["rolling_realized_vol"].min()),
            "max_rolling_vol": float(group["rolling_realized_vol"].max()),
            "mean_state_probability_max": float(group["hmm_state_probability_max"].mean()),
        })
    state_interpretation = pd.DataFrame(rows).sort_values("interpreted_regime_id").reset_index(drop=True)
    interpreted_raw_order = state_interpretation.sort_values("interpreted_regime_id")["raw_hmm_state"].astype(int).tolist()
    transition_matrix_interpreted = model.transmat_[np.ix_(interpreted_raw_order, interpreted_raw_order)]
    return fitted, model, scaler, state_interpretation, transition_matrix_interpreted, valid_mask, X


hmm_asset_data = {}
hmm_models = {}
hmm_scalers = {}
hmm_state_tables = []
hmm_transition_matrices = {}
hmm_valid_masks = {}
hmm_scaled_features = {}
hmm_fitting_log_rows = []

for asset, df in featured_asset_data.items():
    warnings_for_asset = []
    try:
        fitted, model, scaler, state_table, transition_matrix, valid_mask, X = fit_hmm_regimes(
            df,
            feature_cols=features_used_by_asset[asset],
            n_states=N_HMM_STATES,
            covariance_type=HMM_COVARIANCE_TYPE,
            n_iter=HMM_N_ITER,
            random_state=HMM_RANDOM_STATE,
            tol=HMM_TOL,
            scaler_type="standard",
        )
        hmm_asset_data[asset] = fitted
        hmm_models[asset] = model
        hmm_scalers[asset] = scaler
        hmm_transition_matrices[asset] = transition_matrix
        hmm_valid_masks[asset] = valid_mask
        hmm_scaled_features[asset] = X
        state_table = state_table.copy()
        state_table.insert(0, "asset", asset)
        hmm_state_tables.append(state_table)
        converged = bool(getattr(model.monitor_, "converged", False))
        monitor_log_probability = float(model.monitor_.history[-1]) if len(model.monitor_.history) else np.nan
        print(f"{asset}: HMM fitted on {int(valid_mask.sum()):,} rows; converged={converged}; log likelihood={monitor_log_probability:.2f}")
    except Exception as exc:
        warnings_for_asset.append(str(exc))
        warnings.warn(f"{asset}: HMM fitting failed: {exc}")
        continue

    hmm_fitting_log_rows.append({
        "asset": asset,
        "n_rows_raw": int(cleaning_report_df.loc[cleaning_report_df["asset"] == asset, "n_rows_raw"].iloc[0]),
        "n_rows_after_cleaning": int(cleaning_report_df.loc[cleaning_report_df["asset"] == asset, "n_rows_after_cleaning"].iloc[0]),
        "n_rows_used_for_hmm": int(hmm_valid_masks[asset].sum()),
        "features_used": json.dumps(features_used_by_asset[asset]),
        "converged": bool(getattr(hmm_models[asset].monitor_, "converged", False)),
        "monitor_log_probability": float(hmm_models[asset].monitor_.history[-1]) if len(hmm_models[asset].monitor_.history) else np.nan,
        "warnings": "; ".join(warnings_for_asset),
    })

if not hmm_asset_data:
    raise RuntimeError("No assets were successfully processed by the HMM fitting step.")

hmm_state_interpretation_by_asset = pd.concat(hmm_state_tables, ignore_index=True)
hmm_state_interpretation_by_asset.to_csv(TABLE_DIR / "hmm_state_interpretation_by_asset.csv", index=False)
hmm_fitting_log = pd.DataFrame(hmm_fitting_log_rows)
hmm_fitting_log.to_csv(LOG_DIR / "hmm_fitting_log.csv", index=False)
display(hmm_state_interpretation_by_asset)
display(hmm_fitting_log)


C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\.thesis-env\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\.thesis-env\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


ETHUSDT: HMM fitted on 8,736 rows; converged=True; log likelihood=-54601.07


BTCUSDT: HMM fitted on 8,736 rows; converged=True; log likelihood=-52129.35


,asset,raw_hmm_state,interpreted_regime_id,interpreted_regime_label,n_observations,percentage_observations,mean_log_return,std_log_return,mean_abs_log_return,mean_rolling_vol,median_rolling_vol,min_rolling_vol,max_rolling_vol,mean_state_probability_max
0,ETHUSDT,2,0,low_vol,3733,42.731227,0.000262,0.004329,0.003276,0.004384,0.004489,0.001273,0.006822,0.985008
1,ETHUSDT,1,1,medium_vol,3427,39.228480,0.000248,0.005074,0.004044,0.007688,0.007543,0.004219,0.012260,0.957554
2,ETHUSDT,0,2,high_vol,1576,18.040293,-0.001238,0.015210,0.011372,0.011520,0.011249,0.003333,0.032130,0.956566
3,BTCUSDT,2,0,low_vol,3091,35.382326,0.000012,0.002210,0.001704,0.002337,0.002375,0.000657,0.003856,0.981561
4,BTCUSDT,1,1,medium_vol,3909,44.745879,0.000186,0.002988,0.002386,0.004359,0.004260,0.001961,0.007333,0.962339
5,BTCUSDT,0,2,high_vol,1736,19.871795,-0.000485,0.009199,0.007029,0.007146,0.006744,0.002046,0.017149,0.966398


,asset,n_rows_raw,n_rows_after_cleaning,n_rows_used_for_hmm,features_used,converged,monitor_log_probability,warnings
0,ETHUSDT,52577,8760,8736,"[""log_return"", ""abs_log_return"", ""rolling_real...",True,-54601.070683,
1,BTCUSDT,52577,8760,8736,"[""log_return"", ""abs_log_return"", ""rolling_real...",True,-52129.350858,


## 7. HMM State Interpretation Table

The table below proves how raw HMM states were interpreted. Raw state identifiers are arbitrary, so they are sorted by average rolling realized volatility and renamed as low, medium, and high-volatility regimes.


In [6]:
display(hmm_state_interpretation_by_asset)
print(f"Saved: {TABLE_DIR / 'hmm_state_interpretation_by_asset.csv'}")


,asset,raw_hmm_state,interpreted_regime_id,interpreted_regime_label,n_observations,percentage_observations,mean_log_return,std_log_return,mean_abs_log_return,mean_rolling_vol,median_rolling_vol,min_rolling_vol,max_rolling_vol,mean_state_probability_max
0,ETHUSDT,2,0,low_vol,3733,42.731227,0.000262,0.004329,0.003276,0.004384,0.004489,0.001273,0.006822,0.985008
1,ETHUSDT,1,1,medium_vol,3427,39.228480,0.000248,0.005074,0.004044,0.007688,0.007543,0.004219,0.012260,0.957554
2,ETHUSDT,0,2,high_vol,1576,18.040293,-0.001238,0.015210,0.011372,0.011520,0.011249,0.003333,0.032130,0.956566
3,BTCUSDT,2,0,low_vol,3091,35.382326,0.000012,0.002210,0.001704,0.002337,0.002375,0.000657,0.003856,0.981561
4,BTCUSDT,1,1,medium_vol,3909,44.745879,0.000186,0.002988,0.002386,0.004359,0.004260,0.001961,0.007333,0.962339
5,BTCUSDT,0,2,high_vol,1736,19.871795,-0.000485,0.009199,0.007029,0.007146,0.006744,0.002046,0.017149,0.966398


Saved: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\02_hmm_regimes\tables\hmm_state_interpretation_by_asset.csv


## 8. Transition Matrices

The transition matrix describes regime persistence and switching. High diagonal values indicate that the model expects regimes to persist from one observation to the next.


In [7]:
def transition_matrix_to_frame(matrix):
    labels = REGIME_ORDER[: matrix.shape[0]]
    return pd.DataFrame(matrix, index=labels, columns=labels)


transition_long_rows = []
for asset, matrix in hmm_transition_matrices.items():
    frame = transition_matrix_to_frame(matrix)
    path = TABLE_DIR / f"{asset}_hmm_transition_matrix_interpreted.csv"
    frame.to_csv(path, index=True)
    for from_regime in frame.index:
        for to_regime in frame.columns:
            transition_long_rows.append({
                "asset": asset,
                "from_regime": from_regime,
                "to_regime": to_regime,
                "transition_probability": float(frame.loc[from_regime, to_regime]),
            })
    print(f"Saved: {path}")
    display(frame)

hmm_transition_matrix_long = pd.DataFrame(transition_long_rows)
hmm_transition_matrix_long.to_csv(TABLE_DIR / "hmm_transition_matrix_long.csv", index=False)
display(hmm_transition_matrix_long.head(12))


Saved: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\02_hmm_regimes\tables\ETHUSDT_hmm_transition_matrix_interpreted.csv


,low_vol,medium_vol,high_vol
low_vol,0.971583,9.486696e-26,0.028417
medium_vol,0.026284,9.091489e-01,0.064567
high_vol,0.010327,1.903157e-01,0.799357


Saved: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\02_hmm_regimes\tables\BTCUSDT_hmm_transition_matrix_interpreted.csv


,low_vol,medium_vol,high_vol
low_vol,0.966205,0.004215,0.029580
medium_vol,0.023064,0.920034,0.056902
high_vol,0.008901,0.164821,0.826278


,asset,from_regime,to_regime,transition_probability
0,ETHUSDT,low_vol,low_vol,9.715827e-01
1,ETHUSDT,low_vol,medium_vol,9.486696e-26
2,ETHUSDT,low_vol,high_vol,2.841733e-02
3,ETHUSDT,medium_vol,low_vol,2.628438e-02
4,ETHUSDT,medium_vol,medium_vol,9.091489e-01
5,ETHUSDT,medium_vol,high_vol,6.456673e-02
6,ETHUSDT,high_vol,low_vol,1.032727e-02
7,ETHUSDT,high_vol,medium_vol,1.903157e-01
8,ETHUSDT,high_vol,high_vol,7.993570e-01
9,BTCUSDT,low_vol,low_vol,9.662048e-01


## 9. Regime Summary and Run-Length Tables

A regime run is a consecutive block of the same interpreted HMM regime. Run-length diagnostics show whether the HMM creates persistent regimes or rapidly switches labels.


In [8]:
def summarize_hmm_regimes(df, asset):
    valid = df[df["hmm_regime_label"].isin(REGIME_ORDER)].copy()
    rows = []
    for label in REGIME_ORDER:
        group = valid[valid["hmm_regime_label"] == label]
        if group.empty:
            continue
        rows.append({
            "asset": asset,
            "hmm_regime_label": label,
            "n_observations": int(len(group)),
            "percentage_observations": float(len(group) / len(valid) * 100) if len(valid) else np.nan,
            "start_time": group["timestamp"].min(),
            "end_time": group["timestamp"].max(),
            "mean_close": float(group["close"].mean()),
            "mean_log_return": float(group["log_return"].mean()),
            "std_log_return": float(group["log_return"].std()),
            "mean_abs_log_return": float(group["abs_log_return"].mean()),
            "mean_rolling_vol": float(group["rolling_realized_vol"].mean()),
            "median_rolling_vol": float(group["rolling_realized_vol"].median()),
            "min_rolling_vol": float(group["rolling_realized_vol"].min()),
            "max_rolling_vol": float(group["rolling_realized_vol"].max()),
            "mean_state_probability_max": float(group["hmm_state_probability_max"].mean()),
        })
    return pd.DataFrame(rows)


def summarize_hmm_regime_runs(df, asset):
    valid = df[df["hmm_regime_label"].isin(REGIME_ORDER)].copy()
    if valid.empty:
        return pd.DataFrame()
    valid["run_id"] = (valid["hmm_regime_label"] != valid["hmm_regime_label"].shift()).cumsum()
    runs = valid.groupby(["run_id", "hmm_regime_label"], as_index=False).agg(
        run_length=("hmm_regime_label", "size"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
    )
    summary = runs.groupby("hmm_regime_label").agg(
        number_of_runs=("run_length", "count"),
        mean_run_length=("run_length", "mean"),
        median_run_length=("run_length", "median"),
        max_run_length=("run_length", "max"),
    ).reset_index()
    summary.insert(0, "asset", asset)
    return summary.sort_values(["asset", "hmm_regime_label"]).reset_index(drop=True)


hmm_regime_summary_by_asset = pd.concat([summarize_hmm_regimes(df, asset) for asset, df in hmm_asset_data.items()], ignore_index=True)
hmm_regime_run_summary_by_asset = pd.concat([summarize_hmm_regime_runs(df, asset) for asset, df in hmm_asset_data.items()], ignore_index=True)
hmm_regime_summary_by_asset.to_csv(TABLE_DIR / "hmm_regime_summary_by_asset.csv", index=False)
hmm_regime_run_summary_by_asset.to_csv(TABLE_DIR / "hmm_regime_run_summary_by_asset.csv", index=False)
display(hmm_regime_summary_by_asset)
display(hmm_regime_run_summary_by_asset)


,asset,hmm_regime_label,n_observations,percentage_observations,start_time,end_time,mean_close,mean_log_return,std_log_return,mean_abs_log_return,mean_rolling_vol,median_rolling_vol,min_rolling_vol,max_rolling_vol,mean_state_probability_max
0,ETHUSDT,low_vol,3733,42.731227,2025-01-02 00:00:00+00:00,2025-12-31 23:00:00+00:00,3136.401342,0.000262,0.004329,0.003276,0.004384,0.004489,0.001273,0.006822,0.985008
1,ETHUSDT,medium_vol,3427,39.228480,2025-01-07 23:00:00+00:00,2025-12-27 13:00:00+00:00,3092.471083,0.000248,0.005074,0.004044,0.007688,0.007543,0.004219,0.012260,0.957554
2,ETHUSDT,high_vol,1576,18.040293,2025-01-07 14:00:00+00:00,2025-12-26 14:00:00+00:00,2841.913572,-0.001238,0.015210,0.011372,0.011520,0.011249,0.003333,0.032130,0.956566
3,BTCUSDT,low_vol,3091,35.382326,2025-01-03 16:00:00+00:00,2025-12-31 23:00:00+00:00,105200.962714,0.000012,0.002210,0.001704,0.002337,0.002375,0.000657,0.003856,0.981561
4,BTCUSDT,medium_vol,3909,44.745879,2025-01-02 00:00:00+00:00,2025-12-30 08:00:00+00:00,101624.885651,0.000186,0.002988,0.002386,0.004359,0.004260,0.001961,0.007333,0.962339
5,BTCUSDT,high_vol,1736,19.871795,2025-01-02 14:00:00+00:00,2025-12-30 14:00:00+00:00,95394.664240,-0.000485,0.009199,0.007029,0.007146,0.006744,0.002046,0.017149,0.966398


,asset,hmm_regime_label,number_of_runs,mean_run_length,median_run_length,max_run_length
0,ETHUSDT,high_vol,303,5.201320,2.0,122
1,ETHUSDT,low_vol,102,36.598039,27.0,210
2,ETHUSDT,medium_vol,286,11.982517,11.0,39
3,BTCUSDT,high_vol,290,5.986207,2.0,125
4,BTCUSDT,low_vol,97,31.865979,25.0,164
5,BTCUSDT,medium_vol,285,13.715789,12.0,51


## 10. Optional Comparison with Volatility-Quantile Regimes

If the first notebook's quantile labels are available, this section compares transparent threshold-based regimes with data-driven HMM regimes. Perfect agreement is not expected. The quantile method is threshold-based, while the HMM method is persistence-based and feature-driven. Partial disagreement is informative, not a failure.


In [9]:
def load_quantile_labels(asset):
    path = QUANTILE_RESULT_DIR / "regime_labels" / f"{asset}_quantile_regime_labels.parquet"
    if not path.exists():
        warnings.warn(f"{asset}: quantile regime labels not found at {path}")
        return None
    labels = pd.read_parquet(path)
    labels["timestamp"] = convert_timestamp(labels["timestamp"])
    return labels


quantile_labels_by_asset = {}
quantile_labels_found = {}
agreement_summary_rows = []
agreement_counts_by_asset = {}

for asset, hmm_df in hmm_asset_data.items():
    quantile_df = load_quantile_labels(asset)
    quantile_labels_found[asset] = quantile_df is not None
    if quantile_df is None:
        continue
    quantile_labels_by_asset[asset] = quantile_df
    merged = hmm_df.merge(quantile_df[["timestamp", "regime_quantile_id", "regime_quantile_label"]], on="timestamp", how="inner")
    merged = merged[merged["hmm_regime_label"].isin(REGIME_ORDER) & merged["regime_quantile_label"].isin(REGIME_ORDER)].copy()
    if merged.empty:
        warnings.warn(f"{asset}: no overlapping valid HMM and quantile regime labels.")
        continue
    counts = pd.crosstab(merged["hmm_regime_label"], merged["regime_quantile_label"]).reindex(index=REGIME_ORDER, columns=REGIME_ORDER, fill_value=0)
    counts_path = TABLE_DIR / f"{asset}_hmm_vs_quantile_agreement_counts.csv"
    counts.to_csv(counts_path)
    agreement_counts_by_asset[asset] = counts
    same_label_pct = float((merged["hmm_regime_label"] == merged["regime_quantile_label"]).mean() * 100)
    ari_value = np.nan
    if adjusted_rand_score is not None:
        ari_value = float(adjusted_rand_score(merged["hmm_regime_id"].astype(int), merged["regime_quantile_id"].astype(int)))
    agreement_summary_rows.append({
        "asset": asset,
        "n_overlapping_observations": int(len(merged)),
        "same_label_percentage": same_label_pct,
        "adjusted_rand_index": ari_value,
        "counts_table_path": str(counts_path),
    })
    print(f"{asset}: same-label agreement = {same_label_pct:.2f}%; ARI = {ari_value:.4f}")
    display(counts)

hmm_vs_quantile_agreement_summary = pd.DataFrame(agreement_summary_rows)
hmm_vs_quantile_agreement_summary.to_csv(TABLE_DIR / "hmm_vs_quantile_agreement_summary.csv", index=False)
display(hmm_vs_quantile_agreement_summary)


ETHUSDT: same-label agreement = 66.43%; ARI = 0.3830


regime_quantile_label,low_vol,medium_vol,high_vol
hmm_regime_label,,,
low_vol,2782,951,0
medium_vol,76,1701,1650
high_vol,25,231,1320


BTCUSDT: same-label agreement = 71.18%; ARI = 0.3970


regime_quantile_label,low_vol,medium_vol,high_vol
hmm_regime_label,,,
low_vol,2595,496,0
medium_vol,236,2163,1510
high_vol,52,224,1460


,asset,n_overlapping_observations,same_label_percentage,adjusted_rand_index,counts_table_path
0,ETHUSDT,8736,66.426282,0.383025,C:\Users\learn\OneDrive\Desktop\Final Masters ...
1,BTCUSDT,8736,71.176740,0.396970,C:\Users\learn\OneDrive\Desktop\Final Masters ...


## 11. Visualizations

The plots use matplotlib only. They are saved to the HMM result folder for thesis reporting and downstream inspection.


In [10]:
def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_close_price(df, asset):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["timestamp"], df["close"], linewidth=1.0, color="black")
    ax.set_title(f"{asset}: Close Price Over Time")
    ax.set_xlabel("Time")
    ax.set_ylabel("Close price")
    save_figure(fig, FIGURE_DIR / f"{asset}_01_close_price.png")


def plot_price_colored_by_hmm_regime(df, asset):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["timestamp"], df["close"], color="lightgray", linewidth=0.8, label="Close price")
    for label in REGIME_ORDER:
        group = df[df["hmm_regime_label"] == label]
        ax.scatter(group["timestamp"], group["close"], s=7, color=REGIME_COLORS[label], label=label, alpha=0.75)
    ax.set_title(f"{asset}: Close Price Colored by HMM Regime")
    ax.set_xlabel("Time")
    ax.set_ylabel("Close price")
    ax.legend(markerscale=2)
    save_figure(fig, FIGURE_DIR / f"{asset}_02_price_colored_by_hmm_regime.png")


def plot_volatility_colored_by_hmm_regime(df, asset):
    fig, ax = plt.subplots(figsize=(12, 5))
    for label in REGIME_ORDER:
        group = df[df["hmm_regime_label"] == label]
        ax.scatter(group["timestamp"], group["rolling_realized_vol"], s=7, color=REGIME_COLORS[label], label=label, alpha=0.75)
    ax.set_title(f"{asset}: Rolling Volatility Colored by HMM Regime")
    ax.set_xlabel("Time")
    ax.set_ylabel("Rolling realized volatility")
    ax.legend(markerscale=2)
    save_figure(fig, FIGURE_DIR / f"{asset}_03_volatility_colored_by_hmm_regime.png")


def plot_hmm_regime_timeline_strip(df, asset):
    valid = df[df["hmm_regime_id"].notna()].copy()
    if valid.empty:
        warnings.warn(f"{asset}: no valid HMM regimes for timeline strip.")
        return
    y = valid["hmm_regime_id"].astype(int).to_numpy().reshape(1, -1)
    fig, ax = plt.subplots(figsize=(12, 2.2))
    im = ax.imshow(y, aspect="auto", cmap="RdYlGn_r", interpolation="nearest", extent=[0, len(valid), 0, 1], vmin=0, vmax=2)
    ax.set_yticks([])
    ax.set_title(f"{asset}: HMM Regime Timeline Strip")
    ax.set_xlabel("Observation index in analysis window")
    cbar = fig.colorbar(im, ax=ax, orientation="horizontal", pad=0.25, ticks=[0, 1, 2])
    cbar.ax.set_xticklabels(REGIME_ORDER)
    save_figure(fig, FIGURE_DIR / f"{asset}_04_hmm_regime_timeline_strip.png")


def plot_hmm_regime_distribution(df, asset):
    counts = df[df["hmm_regime_label"].isin(REGIME_ORDER)]["hmm_regime_label"].value_counts().reindex(REGIME_ORDER, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(counts.index, counts.values, color=[REGIME_COLORS[label] for label in counts.index])
    ax.set_title(f"{asset}: HMM Regime Distribution")
    ax.set_xlabel("HMM regime")
    ax.set_ylabel("Number of observations")
    for index, value in enumerate(counts.values):
        ax.text(index, value, f"{value:,}", ha="center", va="bottom", fontsize=9)
    save_figure(fig, FIGURE_DIR / f"{asset}_05_hmm_regime_distribution.png")


def plot_hmm_transition_matrix(matrix, asset):
    labels = REGIME_ORDER[: matrix.shape[0]]
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticklabels(labels)
    ax.set_title(f"{asset}: HMM Transition Matrix")
    ax.set_xlabel("To regime")
    ax.set_ylabel("From regime")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", color="white" if value > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    save_figure(fig, FIGURE_DIR / f"{asset}_06_hmm_transition_matrix.png")


def plot_hmm_posterior_confidence(df, asset):
    valid = df[df["hmm_state_probability_max"].notna()].copy()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(valid["timestamp"], valid["hmm_state_probability_max"], color="tab:blue", linewidth=1.0)
    ax.set_ylim(0, 1.02)
    ax.set_title(f"{asset}: HMM Posterior Confidence Over Time")
    ax.set_xlabel("Time")
    ax.set_ylabel("Maximum posterior state probability")
    save_figure(fig, FIGURE_DIR / f"{asset}_07_hmm_posterior_confidence.png")


def plot_feature_distribution_by_hmm_regime(df, asset):
    valid = df[df["hmm_regime_label"].isin(REGIME_ORDER)].copy()
    data = [valid.loc[valid["hmm_regime_label"] == label, "rolling_realized_vol"].dropna().to_numpy() for label in REGIME_ORDER]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.boxplot(data, tick_labels=REGIME_ORDER, showfliers=False)
    ax.set_title(f"{asset}: Rolling Volatility Distribution by HMM Regime")
    ax.set_xlabel("HMM regime")
    ax.set_ylabel("Rolling realized volatility")
    save_figure(fig, FIGURE_DIR / f"{asset}_08_feature_distribution_by_hmm_regime.png")


def plot_hmm_vs_quantile_agreement(counts, asset):
    matrix = counts.reindex(index=REGIME_ORDER, columns=REGIME_ORDER, fill_value=0).to_numpy()
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(matrix, cmap="Purples")
    ax.set_xticks(range(len(REGIME_ORDER)))
    ax.set_yticks(range(len(REGIME_ORDER)))
    ax.set_xticklabels(REGIME_ORDER, rotation=30, ha="right")
    ax.set_yticklabels(REGIME_ORDER)
    ax.set_title(f"{asset}: HMM vs Quantile Agreement")
    ax.set_xlabel("Quantile regime")
    ax.set_ylabel("HMM regime")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{int(matrix[i, j]):,}", ha="center", va="center", color="white" if matrix[i, j] > matrix.max() / 2 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    save_figure(fig, FIGURE_DIR / f"{asset}_09_hmm_vs_quantile_agreement.png")


for asset, df in hmm_asset_data.items():
    plot_close_price(df, asset)
    plot_price_colored_by_hmm_regime(df, asset)
    plot_volatility_colored_by_hmm_regime(df, asset)
    plot_hmm_regime_timeline_strip(df, asset)
    plot_hmm_regime_distribution(df, asset)
    plot_hmm_transition_matrix(hmm_transition_matrices[asset], asset)
    plot_hmm_posterior_confidence(df, asset)
    plot_feature_distribution_by_hmm_regime(df, asset)
    if asset in agreement_counts_by_asset:
        plot_hmm_vs_quantile_agreement(agreement_counts_by_asset[asset], asset)
    print(f"Saved figures for {asset} to {FIGURE_DIR}")


Saved figures for ETHUSDT to C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\02_hmm_regimes\figures


Saved figures for BTCUSDT to C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\02_hmm_regimes\figures


## 12. Auto-Generated Interpretation

This section prints thesis-oriented diagnostics for each asset and explains how to read the HMM outputs.


In [11]:
from IPython.display import Markdown, display


def interpret_hmm_asset(asset):
    df = hmm_asset_data[asset]
    summary = hmm_regime_summary_by_asset[hmm_regime_summary_by_asset["asset"] == asset].copy()
    transition = transition_matrix_to_frame(hmm_transition_matrices[asset])
    valid = df[df["hmm_regime_label"].isin(REGIME_ORDER)]
    pct_lines = []
    vol_lines = []
    for label in REGIME_ORDER:
        row = summary[summary["hmm_regime_label"] == label]
        if row.empty:
            pct_lines.append(f"- {label}: 0.00% of HMM observations")
        else:
            pct_lines.append(f"- {label}: {row['percentage_observations'].iloc[0]:.2f}% of HMM observations")
            vol_lines.append(f"- {label}: mean rolling volatility = {row['mean_rolling_vol'].iloc[0]:.8f}")

    mean_vol = summary.set_index("hmm_regime_label")["mean_rolling_vol"].to_dict()
    high_clear = mean_vol.get("high_vol", np.nan) > mean_vol.get("medium_vol", np.nan) > mean_vol.get("low_vol", np.nan)
    diagonal_values = {label: float(transition.loc[label, label]) for label in transition.index if label in transition.columns}
    persistent = all(value >= 0.50 for value in diagonal_values.values()) if diagonal_values else False
    mean_confidence = float(valid["hmm_state_probability_max"].mean()) if len(valid) else np.nan
    comparison_text = "Quantile comparison was unavailable for this asset."
    if not hmm_vs_quantile_agreement_summary.empty and asset in hmm_vs_quantile_agreement_summary["asset"].values:
        row = hmm_vs_quantile_agreement_summary[hmm_vs_quantile_agreement_summary["asset"] == asset].iloc[0]
        comparison_text = f"Quantile comparison: same-label agreement = {row['same_label_percentage']:.2f}%, adjusted Rand index = {row['adjusted_rand_index']:.4f}."

    display(Markdown(f"""
### {asset} HMM Interpretation

- HMM observations used: {len(valid):,}
{chr(10).join(pct_lines)}

Mean rolling volatility by interpreted HMM regime:

{chr(10).join(vol_lines)}

- High-volatility separation: {'yes, the interpreted high-volatility state has the highest mean rolling volatility' if high_clear else 'not strictly monotonic; inspect state interpretation and feature distribution tables'}.
- Transition matrix diagonal values: {json.dumps(diagonal_values, indent=2)}
- Regime persistence diagnostic: {'persistent regimes' if persistent else 'less persistent or mixed switching behaviour'}.
- Mean posterior confidence: {mean_confidence:.4f}
- {comparison_text}
"""))


for asset in hmm_asset_data:
    interpret_hmm_asset(asset)



### ETHUSDT HMM Interpretation

- HMM observations used: 8,736
- low_vol: 42.73% of HMM observations
- medium_vol: 39.23% of HMM observations
- high_vol: 18.04% of HMM observations

Mean rolling volatility by interpreted HMM regime:

- low_vol: mean rolling volatility = 0.00438383
- medium_vol: mean rolling volatility = 0.00768790
- high_vol: mean rolling volatility = 0.01152014

- High-volatility separation: yes, the interpreted high-volatility state has the highest mean rolling volatility.
- Transition matrix diagonal values: {
  "low_vol": 0.9715826683907139,
  "medium_vol": 0.9091488913870129,
  "high_vol": 0.7993570379906981
}
- Regime persistence diagnostic: persistent regimes.
- Mean posterior confidence: 0.9691
- Quantile comparison: same-label agreement = 66.43%, adjusted Rand index = 0.3830.



### BTCUSDT HMM Interpretation

- HMM observations used: 8,736
- low_vol: 35.38% of HMM observations
- medium_vol: 44.75% of HMM observations
- high_vol: 19.87% of HMM observations

Mean rolling volatility by interpreted HMM regime:

- low_vol: mean rolling volatility = 0.00233732
- medium_vol: mean rolling volatility = 0.00435910
- high_vol: mean rolling volatility = 0.00714581

- High-volatility separation: yes, the interpreted high-volatility state has the highest mean rolling volatility.
- Transition matrix diagonal values: {
  "low_vol": 0.9662048333030723,
  "medium_vol": 0.9200340206692711,
  "high_vol": 0.8262783685663152
}
- Regime persistence diagnostic: persistent regimes.
- Mean posterior confidence: 0.9699
- Quantile comparison: same-label agreement = 71.18%, adjusted Rand index = 0.3970.


### How to Read the HMM Outputs

- The price-colored plot shows when inferred regimes occur along the market price path.
- The volatility-colored plot checks whether the high-volatility HMM state corresponds to visibly higher rolling volatility.
- The transition matrix shows persistence and switching between interpreted regimes.
- Posterior confidence shows how certain the model is about its most likely state at each timestamp.
- The feature distribution plot validates whether rolling volatility separates the interpreted states.
- The HMM vs quantile heatmap shows the relationship between transparent threshold regimes and data-driven probabilistic regimes.


## 13. Save HMM Regime Labels and Model Artifacts

This section saves clean label files, fitted HMM objects, scalers, and feature-column metadata. The initial rows without enough historical data for rolling features are retained as `insufficient_history` so timestamps remain aligned with the cleaned price series.


In [12]:
label_output_paths = {}
model_output_paths = {}
base_label_columns = [
    "timestamp",
    "close",
    "log_return",
    "abs_log_return",
    "rolling_realized_vol",
    "rolling_mean_return",
    "rolling_abs_return_mean",
    "raw_hmm_state",
    "hmm_regime_id",
    "hmm_regime_label",
    "hmm_state_probability_max",
]
prob_columns = [f"hmm_prob_state_{state}" for state in range(N_HMM_STATES)]
volume_feature_columns = ["volume", "log_volume", "volume_log_change", "rolling_volume_mean", "rolling_volume_std", "rolling_volume_z"]

for asset, df in hmm_asset_data.items():
    label_columns = base_label_columns + prob_columns
    label_columns += [col for col in volume_feature_columns if col in df.columns and col not in label_columns]
    labels = df[label_columns].copy()
    parquet_path = REGIME_LABEL_DIR / f"{asset}_hmm_regime_labels.parquet"
    csv_path = REGIME_LABEL_DIR / f"{asset}_hmm_regime_labels.csv"
    labels.to_parquet(parquet_path, index=False)
    labels.to_csv(csv_path, index=False)
    label_output_paths[asset] = {"parquet": str(parquet_path), "csv": str(csv_path)}

    model_path = MODEL_DIR / f"{asset}_hmm_model.pkl"
    scaler_path = MODEL_DIR / f"{asset}_hmm_scaler.pkl"
    feature_path = MODEL_DIR / f"{asset}_hmm_feature_columns.json"
    with open(model_path, "wb") as file:
        pickle.dump(hmm_models[asset], file)
    with open(scaler_path, "wb") as file:
        pickle.dump(hmm_scalers[asset], file)
    with open(feature_path, "w", encoding="utf-8") as file:
        json.dump(features_used_by_asset[asset], file, indent=2)
    model_output_paths[asset] = {"model": str(model_path), "scaler": str(scaler_path), "feature_columns": str(feature_path)}
    print(f"Saved labels and model artifacts for {asset}")


Saved labels and model artifacts for ETHUSDT
Saved labels and model artifacts for BTCUSDT


## 14. Optional Model Selection Appendix

The final thesis model remains fixed at three states for interpretability: low, medium, and high volatility. The diagnostics below fit 2, 3, and 4 states when feasible and report likelihood-based approximations for comparison.


In [13]:
def approximate_hmm_parameter_count(n_states, n_features, covariance_type):
    start_params = n_states - 1
    transition_params = n_states * (n_states - 1)
    mean_params = n_states * n_features
    if covariance_type == "full":
        covariance_params = n_states * n_features * (n_features + 1) / 2
    elif covariance_type == "diag":
        covariance_params = n_states * n_features
    elif covariance_type == "spherical":
        covariance_params = n_states
    elif covariance_type == "tied":
        covariance_params = n_features * (n_features + 1) / 2
    else:
        covariance_params = n_states * n_features
    return int(start_params + transition_params + mean_params + covariance_params)


model_selection_rows = []
for asset, X in hmm_scaled_features.items():
    n_features = X.shape[1]
    for n_states in [2, 3, 4]:
        try:
            model = GaussianHMM(
                n_components=n_states,
                covariance_type=HMM_COVARIANCE_TYPE,
                n_iter=min(HMM_N_ITER, 200),
                random_state=HMM_RANDOM_STATE,
                tol=HMM_TOL,
                min_covar=1e-6,
            )
            model.fit(X)
            log_likelihood = float(model.score(X))
            k = approximate_hmm_parameter_count(n_states, n_features, HMM_COVARIANCE_TYPE)
            model_selection_rows.append({
                "asset": asset,
                "n_states": n_states,
                "n_observations": int(X.shape[0]),
                "n_features": int(n_features),
                "log_likelihood": log_likelihood,
                "parameter_count_approx": int(k),
                "aic_approx": float(2 * k - 2 * log_likelihood),
                "bic_approx": float(np.log(X.shape[0]) * k - 2 * log_likelihood),
                "converged": bool(getattr(model.monitor_, "converged", False)),
                "warning": "",
            })
        except Exception as exc:
            warnings.warn(f"{asset}: model selection failed for {n_states} states: {exc}")
            model_selection_rows.append({
                "asset": asset,
                "n_states": n_states,
                "n_observations": int(X.shape[0]),
                "n_features": int(n_features),
                "log_likelihood": np.nan,
                "parameter_count_approx": np.nan,
                "aic_approx": np.nan,
                "bic_approx": np.nan,
                "converged": False,
                "warning": str(exc),
            })

hmm_model_selection_diagnostics = pd.DataFrame(model_selection_rows)
hmm_model_selection_diagnostics.to_csv(TABLE_DIR / "hmm_model_selection_diagnostics.csv", index=False)
display(hmm_model_selection_diagnostics)


,asset,n_states,n_observations,n_features,log_likelihood,parameter_count_approx,aic_approx,bic_approx,converged,warning
0,ETHUSDT,2,8736,7,-59408.326968,73,118962.653937,119479.144099,True,
1,ETHUSDT,3,8736,7,-54601.070621,113,109428.141242,110227.639712,True,
2,ETHUSDT,4,8736,7,-4925.950505,155,10161.901009,11258.558202,True,
3,BTCUSDT,2,8736,7,-57401.516983,73,114949.033966,115465.524128,True,
4,BTCUSDT,3,8736,7,-52129.350819,113,104484.701637,105284.200107,True,
5,BTCUSDT,4,8736,7,-7411.937792,155,15133.875584,16230.532777,True,


## 15. Save Configuration and Logs

The configuration records HMM settings, feature columns, input files, quantile-label availability, and output paths.


In [14]:
config = {
    "target_symbols": TARGET_SYMBOLS,
    "assets_processed": list(hmm_asset_data.keys()),
    "use_year_filter": USE_YEAR_FILTER,
    "year_analyzed": YEAR_TO_ANALYZE,
    "n_hmm_states": N_HMM_STATES,
    "covariance_type": HMM_COVARIANCE_TYPE,
    "n_iter": HMM_N_ITER,
    "random_state": HMM_RANDOM_STATE,
    "tolerance": HMM_TOL,
    "feature_columns_used_per_asset": features_used_by_asset,
    "rolling_window_used_per_asset": rolling_windows_used,
    "input_files_used": input_files_used,
    "quantile_labels_found": quantile_labels_found,
    "output_directory": str(RESULT_DIR),
    "created_timestamp": datetime.now(timezone.utc).isoformat(),
    "no_future_looking_features_used": True,
    "feature_engineering_note": "All rolling features use trailing windows over current and historical values only. No centered rolling windows or future values are used.",
}

config_path = CONFIG_DIR / "hmm_regime_config.json"
with open(config_path, "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)
discovery_log.to_csv(LOG_DIR / "data_discovery_log.csv", index=False)
hmm_fitting_log.to_csv(LOG_DIR / "hmm_fitting_log.csv", index=False)
print(f"Saved config: {config_path}")
print(f"Saved discovery log: {LOG_DIR / 'data_discovery_log.csv'}")
print(f"Saved fitting log: {LOG_DIR / 'hmm_fitting_log.csv'}")


Saved config: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\02_hmm_regimes\config\hmm_regime_config.json
Saved discovery log: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\02_hmm_regimes\logs\data_discovery_log.csv
Saved fitting log: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\02_hmm_regimes\logs\hmm_fitting_log.csv


## 16. Thesis-Ready Conclusion

This notebook implemented Hidden Markov Model regime detection as the data-driven probabilistic regime method for the thesis. Unlike volatility-quantile segmentation, the HMM infers latent market states from return and volatility-related features and captures persistence through a transition matrix. The raw HMM states were interpreted by sorting them according to average rolling realized volatility, producing low, medium, and high-volatility regimes. These HMM labels will be used in later notebooks to compare regime-agnostic motif discovery against regime-conditioned motif discovery using Matrix Profile and LoCoMotif.

**What worked**

- HMM produced interpretable regime labels.
- The transition matrix gives persistence evidence.
- Posterior probabilities give confidence evidence.
- Labels are saved for downstream motif mining.

**Limitations**

- HMM state labels are model-dependent.
- The result depends on feature scaling and number of states.
- Gaussian emissions are a simplification.
- HMM is not a supervised market-crisis classifier.
- External validation with VIX can be added later for equity/index data.

**Next step**

- Compare quantile regimes vs HMM regimes.
- Use both regime label sets for regime-conditioned Matrix Profile and LoCoMotif motif mining.


## 17. Final Validation

The final cell checks that the notebook produced labels, summary tables, transition tables, figures, model artifacts, config, and non-empty interpreted regimes. It prints the success message only if all required checks pass.


In [15]:
def validate_hmm_regime_notebook():
    checks = []
    warnings_list = []
    checks.append((len(hmm_asset_data) >= 1, "At least one asset processed"))
    checks.append(((TABLE_DIR / "hmm_regime_summary_by_asset.csv").exists(), "HMM regime summary table exists"))
    checks.append(((TABLE_DIR / "hmm_regime_run_summary_by_asset.csv").exists(), "HMM run summary table exists"))
    checks.append(((TABLE_DIR / "hmm_transition_matrix_long.csv").exists(), "HMM transition matrix long table exists"))
    checks.append(((TABLE_DIR / "hmm_state_interpretation_by_asset.csv").exists(), "HMM state interpretation table exists"))
    checks.append(((LOG_DIR / "data_discovery_log.csv").exists(), "Data discovery log exists"))
    checks.append(((LOG_DIR / "hmm_fitting_log.csv").exists(), "HMM fitting log exists"))
    checks.append(((CONFIG_DIR / "hmm_regime_config.json").exists(), "Config JSON exists"))
    checks.append((config.get("no_future_looking_features_used") is True, "No future-looking features were used"))

    for asset, df in hmm_asset_data.items():
        checks.append(((REGIME_LABEL_DIR / f"{asset}_hmm_regime_labels.parquet").exists(), f"Parquet labels exist for {asset}"))
        checks.append(((REGIME_LABEL_DIR / f"{asset}_hmm_regime_labels.csv").exists(), f"CSV labels exist for {asset}"))
        checks.append(((MODEL_DIR / f"{asset}_hmm_model.pkl").exists(), f"HMM model file exists for {asset}"))
        checks.append(((MODEL_DIR / f"{asset}_hmm_scaler.pkl").exists(), f"HMM scaler file exists for {asset}"))
        checks.append(((MODEL_DIR / f"{asset}_hmm_feature_columns.json").exists(), f"HMM feature-column JSON exists for {asset}"))
        checks.append(((TABLE_DIR / f"{asset}_hmm_transition_matrix_interpreted.csv").exists(), f"Transition matrix table exists for {asset}"))
        figure_count = len(list(FIGURE_DIR.glob(f"{asset}_*.png")))
        checks.append((figure_count >= 6, f"At least 6 figures exist for {asset}"))
        valid_counts = df[df["hmm_regime_label"].isin(REGIME_ORDER)]["hmm_regime_label"].value_counts()
        non_empty_regimes = int((valid_counts.reindex(REGIME_ORDER, fill_value=0) > 0).sum())
        checks.append((non_empty_regimes >= 2, f"{asset} has at least two non-empty HMM regimes"))
        if non_empty_regimes < 3:
            warnings_list.append(f"{asset}: fewer than all three HMM regimes are present; inspect model diagnostics.")
        feature_cols = features_used_by_asset.get(asset, [])
        forbidden_terms = ["lead", "future", "centered"]
        checks.append((not any(any(term in col.lower() for term in forbidden_terms) for col in feature_cols), f"{asset} feature names do not indicate future-looking features"))

    validation_df = pd.DataFrame(checks, columns=["passed", "check"])
    display(validation_df)
    for warning_message in warnings_list:
        warnings.warn(warning_message)
    failed = validation_df[~validation_df["passed"]]
    if not failed.empty:
        raise AssertionError("Validation failed:\n" + failed.to_string(index=False))
    print("HMM REGIME NOTEBOOK COMPLETED SUCCESSFULLY")


validate_hmm_regime_notebook()


,passed,check
0,True,At least one asset processed
1,True,HMM regime summary table exists
2,True,HMM run summary table exists
3,True,HMM transition matrix long table exists
4,True,HMM state interpretation table exists
5,True,Data discovery log exists
6,True,HMM fitting log exists
7,True,Config JSON exists
8,True,No future-looking features were used
9,True,Parquet labels exist for ETHUSDT


HMM REGIME NOTEBOOK COMPLETED SUCCESSFULLY
